# smp_jetmass_run2 — interactive run / parameter tuning

Tune parameters below, **lock** them (writes `configs/last_run.json`), then run any
of the three channels (zjet / dijet / trijet) via the shared
`notebook_utils.run_from_config` — the same path the CLI uses
(`python scripts/run_analysis_cli.py --config configs/last_run.json`).

**Hadronic tip:** for a quick `test` run, set the **HT filter** to a high-HT bin
(e.g. `HT1000to1500`) — the low-HT bins (HT200to300, ...) select no dijet/trijet
events, so a test on them is vacuous.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys, importlib
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "smp_jetmass_run2").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import smp_jetmass_run2.notebook_utils as nbutils
importlib.reload(nbutils)

CONFIG_FILE = repo_root / "configs" / "last_run.json"
DEFAULTS = dict(nbutils.ANALYSIS_CONFIG_DEFAULTS)

DATASETS = {
    "zjet":   ["pythia", "data", "herwig", "powheg", "st", "backgrounds", "pythia_local", "pythia2"],
    "dijet":  ["pythia", "mg_pythia8", "herwig", "data"],
    "trijet": ["pythia", "mg_pythia8", "herwig", "data"],
}
def _hadronic_modes(channel):
    excluded = nbutils.HADRONIC_CHANNEL_EXCLUDED_MODES.get(channel, set())
    return sorted(nbutils.HADRONIC_MODES - excluded)
MODES = {
    "zjet":   ["minimal", "minimal_rho", "validation", "full", "mass_jk", "rho_jk", "reweight_pythia", "reweight_pythia_rho", "reweight_data_prior_rho"],
    "dijet":  _hadronic_modes("dijet"),
    "trijet": _hadronic_modes("trijet"),
}
print("repo_root:", repo_root)

In [ ]:
# -------------------- widgets --------------------
def _load():
    cfg = DEFAULTS.copy()
    if CONFIG_FILE.exists():
        try: cfg.update(json.loads(CONFIG_FILE.read_text()))
        except Exception as e: print("Could not read", CONFIG_FILE, e)
    return cfg
cfg0 = _load()

channel_w  = widgets.Dropdown(options=nbutils.CHANNEL_OPTIONS, value=cfg0["channel"], description="channel")
dataset_w  = widgets.Dropdown(options=DATASETS[cfg0["channel"]], value=cfg0["dataset"] if cfg0["dataset"] in DATASETS[cfg0["channel"]] else DATASETS[cfg0["channel"]][0], description="dataset")
mode_w     = widgets.Dropdown(options=MODES[cfg0["channel"]], value=cfg0["mode"] if cfg0["mode"] in MODES[cfg0["channel"]] else MODES[cfg0["channel"]][0], description="mode")
era_w      = widgets.Dropdown(options=["2016APV","2016","2017","2018","all"], value=cfg0["era"], description="era")
syst_w     = widgets.Dropdown(options=["all_syst","minimal_syst","no_syst"], value=cfg0["systematic_profile"], description="systs")
executor_w = widgets.Dropdown(options=nbutils.EXECUTOR_MODE_OPTIONS, value=cfg0["executor_mode"] if cfg0["executor_mode"] in nbutils.EXECUTOR_MODE_OPTIONS else "futures", description="executor")
dfilter_w  = widgets.Text(value=cfg0.get("dataset_filter",""), description="HT filter", placeholder="hadronic, e.g. HT1000to1500")
casa_w     = widgets.Checkbox(value=cfg0["casa"], description="casa")
test_w     = widgets.Checkbox(value=cfg0["test"], description="test (1 file/chunk)")
useDefault_w=widgets.Checkbox(value=cfg0["useDefault"], description="useDefault client")
chunksize_w= widgets.IntText(value=cfg0["chunksize"], description="chunksize")
chunktest_w= widgets.IntText(value=cfg0["chunksize_test"], description="chunk(test)")
group_w    = widgets.Dropdown(options=["all_in_one","per_group"], value=cfg0["group_mode"], description="group_mode")
redir_w    = widgets.Dropdown(options=nbutils.REDIRECTOR_OPTIONS, value=cfg0.get("redirector","casa") if cfg0.get("redirector","casa") in nbutils.REDIRECTOR_OPTIONS else "casa", description="redirector")
wmem_w     = widgets.Text(value=cfg0.get("worker_memory") or "", description="worker mem", placeholder="blank = 6 GiB default")
wmerge_w   = widgets.IntText(value=int(cfg0.get("worker_merge", 1) or 1), description="wrk merge")

def _on_channel(change):
    ch = change["new"]
    dataset_w.options = DATASETS[ch]
    if dataset_w.value not in DATASETS[ch]: dataset_w.value = DATASETS[ch][0]
    mode_w.options = MODES[ch]
    if mode_w.value not in MODES[ch]: mode_w.value = MODES[ch][0]
    dfilter_w.disabled = ch not in ("dijet", "trijet")
channel_w.observe(_on_channel, names="value")
dfilter_w.disabled = channel_w.value not in ("dijet", "trijet")

display(widgets.VBox([
    widgets.HBox([channel_w, dataset_w, mode_w]),
    widgets.HBox([era_w, syst_w, executor_w]),
    widgets.HBox([dfilter_w, casa_w, test_w, useDefault_w]),
    widgets.HBox([chunksize_w, chunktest_w, group_w, redir_w]),
    widgets.HBox([wmem_w, wmerge_w]),
]))

In [ ]:
# -------------------- lock parameters --------------------
def get_config():
    redirector = redir_w.value
    cfg = {
        "channel": channel_w.value, "dataset": dataset_w.value, "mode": mode_w.value,
        "era": era_w.value, "systematic_profile": syst_w.value,
        "executor_mode": executor_w.value, "casa": casa_w.value, "test": test_w.value,
        "useDefault": useDefault_w.value, "chunksize": int(chunksize_w.value),
        "chunksize_test": int(chunktest_w.value), "group_mode": group_w.value,
        "redirector": redirector, "prependstr": nbutils.resolve_redirector_prepend(redirector),
        "dataset_filter": dfilter_w.value.strip(),
        # streaming-executor knobs (worker_merge=1 = merge nothing on workers;
        # blank worker mem = cluster default 6 GiB)
        "worker_merge": int(wmerge_w.value or 1),
    }
    if wmem_w.value.strip():
        cfg["worker_memory"] = wmem_w.value.strip()
    return nbutils.validate_analysis_config(cfg)

cfg = get_config()
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(json.dumps(cfg, indent=2))
print("Locked config ->", CONFIG_FILE)
print(json.dumps(cfg, indent=2))

In [ ]:
# -------------------- run --------------------
# Same path as the CLI. run_from_config builds the fileset (honouring the HT
# filter for hadronic), runs, and saves under outputs/.
outputs, out = nbutils.run_from_config(cfg, repo_root=repo_root)
print("outputs:", outputs)

In [ ]:
# -------------------- inspect --------------------
if out is not None:
    for k, v in out.items():
        if hasattr(v, "axes"):
            print(f"{k}: axes={[a.name for a in v.axes]}")

In [ ]:
# example: project a reco hist (mass mode) and draw
import matplotlib.pyplot as plt
if out is not None and "ptjet_mjet_g_reco" in out:
    out["ptjet_mjet_g_reco"].project("mreco").plot()
    plt.title(f"{cfg['channel']} groomed jet mass (reco)")
    plt.show()

In [ ]:
# groomed jet rho (reco), pt-bin wise
import numpy as np
import matplotlib.pyplot as plt
if out is not None and "ptjet_rhojet_g_reco" in out:
    h = out["ptjet_rhojet_g_reco"]
    pt_ax = h.axes["ptreco"]
    edges = pt_ax.edges
    n = pt_ax.size
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5), sharey=True)
    axes = np.atleast_1d(axes)
    for i in range(n):
        h[{"ptreco": i}].project("mpt_reco").plot(ax=axes[i])
        axes[i].set_title(f"{edges[i]:.0f}–{edges[i + 1]:.0f} GeV")
    fig.suptitle(f"{cfg['channel']} groomed jet rho (reco), per pt bin")
    plt.tight_layout()
    plt.show()

In [ ]:
# -------------------- DATA orthogonality check (optional) --------------------
# Run the same DATA config for two/three channels, then compare finally-selected
# events. Example (after running dijet & trijet data and keeping their outputs):
#   from smp_jetmass_run2.notebook_utils import channel_overlap
#   channel_overlap({"dijet": out_dijet, "trijet": out_trijet})